# LRN Training Example

This notebook trains an `LRNModel` on the multilayer SIMNRA datasets generated by `examples/generate_multilayer.py`.

The workflow is:

1. Load every `layers_*` dataset case.
2. Pad each case into the open-parameter layout of the max-layer dataset.
3. Build an LRN schema from that max-layer input spec.
4. Train the network on one selected output method such as `RBS`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys 
sys.path.append("../")

import matplotlib.pyplot as plt
import numpy as np
import torch

from ibamlkit.data import DatasetBatchReader
from ibamlkit.models.forward import LRNModel, LTNModel, build_lrn_model_schema
from ibamlkit.training import (
    ConstantFactorTransform,
    EpochSchedule,
    IdentityTransform,
    Chi2Loss,
    MinMaxScaler,
    PeakAwareLoss,
    SupervisedTrainer,
    TransformPipeline,
    prepare_variable_layer_surrogate_dataset,
    shuffle_in_unison,
    split_train_val_test,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ibamlkit").exists() and (candidate / "examples").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


def find_dataset_root(repo_root: Path) -> Path:
    candidates = [
        repo_root / "datasets" / "multilayer_14_elements",
        repo_root / "examples" / "datasets" / "multilayer_14_elements",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find datasets/multilayer_14_elements. Run examples/generate_multilayer.py first."
    )


def parse_layer_count(path: Path) -> int:
    match = re.search(r"layers_(\d+)$", path.name)
    if match is None:
        raise ValueError(f"Could not parse layer count from {path}")
    return int(match.group(1))


def collect_case_dirs(dataset_root: Path) -> list[Path]:
    case_dirs = [path for path in dataset_root.iterdir() if path.is_dir() and path.name.startswith("layers_")]
    if not case_dirs:
        raise FileNotFoundError(f"No layers_* directories found in {dataset_root}")
    return sorted(case_dirs, key=parse_layer_count)


def load_case_dataset(case_dir: Path):
    reader = DatasetBatchReader()
    paths = reader.collect_dataset_paths(case_dir)
    dataset = reader.load_many(paths)
    return dataset, paths


def print_array_stats(name: str, x: np.ndarray) -> None:
    x = np.asarray(x, dtype=np.float64)
    print(
        f"{name}: shape={x.shape}, min={x.min():.6g}, max={x.max():.6g}, "
        f"mean={x.mean():.6g}, median={np.median(x):.6g}"
    )


In [ ]:
repo_root = find_repo_root()
dataset_root = find_dataset_root(repo_root)
method_name = "RBS"
target_width = 4000
target_scale_factor = 1e-6
seed = 7
val_count = 5000
test_count = 5000

case_dirs = collect_case_dirs(dataset_root)
datasets = []
for case_dir in case_dirs:
    try :
        dataset, paths = load_case_dataset(case_dir)
        n_layers = int(dataset.input_spec.generation_info.get("n_layers", 0))
        print(f"Loaded layer case {n_layers}: {len(paths)} file(s), {dataset.sample_count} samples")
        datasets.append(dataset)
    except Exception as e:        print(f"Error loading case {case_dir}: {e}")

reference_dataset = max(
    datasets,
    key=lambda dataset: int(dataset.input_spec.generation_info.get("n_layers", 0)),
)
full_target_width = int(reference_dataset.spectra[method_name].shape[1])
model_target_width = full_target_width if target_width is None else min(int(target_width), full_target_width)
schema = build_lrn_model_schema(
    reference_dataset.input_spec,
    model_name=f"lrn_{method_name.lower()}",
    task_method_names=[method_name],
    output_spectra_lengths={method_name: model_target_width},
)
prepared = prepare_variable_layer_surrogate_dataset(
    datasets,
    schema=schema,
    method_name=method_name,
)
reference_dataset = prepared.reference_dataset
x = prepared.inputs_selected
y_raw = prepared.targets[:, :model_target_width]
y_lengths = prepared.target_lengths
if y_lengths is not None:
    y_lengths = np.minimum(y_lengths, model_target_width)

output_transform_steps = [IdentityTransform()]
if target_scale_factor is not None:
    output_transform_steps.append(ConstantFactorTransform(target_scale_factor))
output_transform = TransformPipeline(output_transform_steps)
y = output_transform.fit_transform(y_raw)

print_array_stats("Raw targets", y_raw)
print_array_stats("Transformed targets", y)

print("Dataset root:", dataset_root)
print("Reference layer count:", reference_dataset.input_spec.generation_info.get("n_layers"))
print("Input matrix shape:", x.shape)
print("Output matrix shape:", y.shape)
print("Schema input dimension:", schema.inputs.dimension)
print("Schema output width:", schema.outputs.spectra_lengths[method_name])
print("Full output width:", full_target_width)

In [ ]:
plt.hist(dataset.open_parameter_values[:,0 ])

In [ ]:
x, y, y_lengths = shuffle_in_unison(x, y, seed=seed, target_lengths=y_lengths)
split = split_train_val_test(
    x,
    y,
    val_count=val_count,
    test_count=test_count,
    target_lengths=y_lengths,
)

x_train = split.train_inputs
y_train = split.train_targets
x_val = split.val_inputs
y_val = split.val_targets
x_test = split.test_inputs
y_test = split.test_targets
test_lengths = split.test_target_lengths

input_scaler = MinMaxScaler(low=0.0, high=1.0)
x_train_scaled = input_scaler.fit_transform(x_train)
x_val_scaled = input_scaler.transform(x_val)
x_test_scaled = input_scaler.transform(x_test)

print("Train:", x_train_scaled.shape, y_train.shape)
print("Val:", x_val_scaled.shape, y_val.shape)
print("Test:", x_test_scaled.shape, y_test.shape)

In [ ]:
test_lengths 

In [ ]:
for i in range(3) : 
    plt.plot(y_train[i], label="Train")


In [ ]:

model = LTNModel(
    schema,
    model_dim=256,
    num_heads=1,
    num_encoder_layers=2,
    feedforward_dim=768,
    dropout=0.1,
    decoder_hidden_sizes=(512, 512),
    refiner_hidden_channels=32,
    refiner_kernel_size=17,
)

modell = LRNModel(
    schema,
    hidden_size= 256,
    contribution_size= 256,
    setup_embedding_dim = 32,
    layer_embedding_dim = 256,
    block_hidden_sizes = (512, 512),
    decoder_hidden_sizes = (768, 768),
    refiner_hidden_channels=32,
    refiner_kernel_size=17,
)


trainer = SupervisedTrainer(
    device=device,
    loss_fn=PeakAwareLoss(
        amplitude_weight=1.0,
        gradient_weight=0.25,
        curvature_weight=0.05,
    ),
    optimizer_name="adamw",
    weight_decay=1e-3,
    max_grad_norm=1.0,
    early_stopping_patience=10,
    verbose=True,
    log_every_epochs=1,
)

result = trainer.fit(
    model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        #EpochSchedule(learning_rate=5e-3, epochs=10, batch_size=256),
        EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print(result)

In [ ]:
pred_test_transformed = model.predict(x_test_scaled).cpu().numpy()
pred_test = output_transform.inverse_transform(pred_test_transformed)
y_test_original = output_transform.inverse_transform(y_test)

print_array_stats("Predicted test targets", pred_test)
chi2 = np.mean((pred_test - y_test_original) ** 2 / (y_test_original + 1.0), axis=1)
print("Test mean chi2:", float(np.mean(chi2)))
print("Test median chi2:", float(np.median(chi2)))

n_plot = 50
fig, axes = plt.subplots(n_plot, 1, figsize=(7, 1.5 * n_plot), sharex=False)
if n_plot == 1:
    axes = [axes]

for row_index in range(n_plot):
    true_length = y_test_original.shape[1] if test_lengths is None else min(int(test_lengths[row_index]), y_test_original.shape[1])
    axes[row_index].plot(y_test_original[row_index, :true_length], label="target")
    axes[row_index].plot(pred_test[row_index, :true_length], linestyle="--", label="prediction")
    axes[row_index].set_ylabel(f"sample {row_index}")
    if row_index == 0:
        axes[row_index].legend()

axes[-1].set_xlabel("channel")
plt.tight_layout()
plt.show()

In [ ]:
artifact_dir = repo_root / "examples" / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"lrn_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
        "training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)